# Ellipsoid Fitting via Least Squares with Ellipsoid-Specific Constraints
## Reproducible Workflow — Li & Griffiths (2004)

This notebook demonstrates the Python implementation of the ellipsoid-specific least-squares fitting
algorithm described in:

> Li, Q. and Griffiths, J. G. (2004).  
> *Least squares ellipsoid specific fitting.*  
> Proceedings of the Geometric Modeling and Processing, 2004. IEEE, pp. 335–340.  
> DOI: [10.1109/GMAP.2004.1290055](https://doi.org/10.1109/GMAP.2004.1290055)

### Contents
1. [Install & import](#1-install--import)
2. [Synthetic data generation](#2-synthetic-data-generation)
3. [Fitting an axis-aligned ellipsoid](#3-fitting-an-axis-aligned-ellipsoid)
4. [Fitting a rotated ellipsoid](#4-fitting-a-rotated-ellipsoid)
5. [Fitting from CSV datasets](#5-fitting-from-csv-datasets)
6. [Accuracy vs noise study](#6-accuracy-vs-noise-study)
7. [Interactive 3D Visualisation](#7-interactive-3d-visualisation)
8. [Fitting Accuracy Metrics](#8-fitting-accuracy-metrics)
9. [Interactive Controls (ipywidgets)](#9-interactive-controls-ipywidgets)
10. [How to Interpret the Plot and Metrics](#10-how-to-interpret-the-plot-and-metrics)

## 1. Install & import

In [ ]:
# If running from the repository root, the package is importable directly.
# Otherwise: pip install -e .
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib
# NOTE: Remove or change the backend below when running interactively in Jupyter.
# Use matplotlib.use('Agg') for headless/CI environments, or omit for inline display.
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        # Running inside a Jupyter notebook — use inline backend
        _ip.run_line_magic('matplotlib', 'inline')
    else:
        matplotlib.use('Agg')  # headless / CI
except Exception:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Plotly for interactive 3-D visualisation
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    _PLOTLY_AVAILABLE = True
except ImportError:
    _PLOTLY_AVAILABLE = False
    print("Plotly not installed. Run: pip install plotly")

# ipywidgets for interactive controls
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    _WIDGETS_AVAILABLE = True
except ImportError:
    _WIDGETS_AVAILABLE = False
    print("ipywidgets not installed. Run: pip install ipywidgets")

from ellipsoid_fitting import (
    fit_ellipsoid,
    generate_ellipsoid_points,
    algebraic_distance,
    residuals_rms,
)
print("ellipsoid_fitting imported successfully")
print(f"Plotly available : {_PLOTLY_AVAILABLE}")
print(f"ipywidgets available: {_WIDGETS_AVAILABLE}")

## 2. Synthetic data generation

The `generate_ellipsoid_points` function creates quasi-uniformly distributed points on an ellipsoid surface using the Fibonacci sphere algorithm, optionally perturbed with Gaussian noise.

In [ ]:
TRUE_CENTRE = (1.0, 2.0, 3.0)
TRUE_RADII  = (5.0, 3.0, 2.0)

pts = generate_ellipsoid_points(
    centre=TRUE_CENTRE,
    radii=TRUE_RADII,
    n_points=300,
    noise_std=0.05,
    seed=42,
)

print(f"Shape : {pts.shape}")
print(f"x range: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"y range: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"z range: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

## 3. Fitting an axis-aligned ellipsoid

In [ ]:
x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
result = fit_ellipsoid(x, y, z)

print("=" * 55)
print("Li–Griffiths Ellipsoid Fit")
print("=" * 55)
print(f"True  centre : {np.array(TRUE_CENTRE)}")
print(f"Fitted centre: {result['centre'].round(4)}")
print()
print(f"True  radii  : {np.sort(np.array(TRUE_RADII))[::-1]}")
print(f"Fitted radii : {result['radii'].round(4)}")
print()
print(f"RMS algebraic residual: {residuals_rms(x, y, z, result):.6f}")
print("=" * 55)

## 4. Fitting a rotated ellipsoid

The algorithm handles arbitrary orientations without modification.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler('xyz', [30, 45, 60], degrees=True).as_matrix()
pts_rot = generate_ellipsoid_points(
    centre=(0.0, 0.0, 0.0),
    radii=(6.0, 4.0, 2.5),
    rotation=R,
    n_points=400,
    noise_std=0.10,
    seed=7,
)

result_rot = fit_ellipsoid(pts_rot[:, 0], pts_rot[:, 1], pts_rot[:, 2])
print(f"True  radii : {np.sort([6.0, 4.0, 2.5])[::-1]}")
print(f"Fitted radii: {result_rot['radii'].round(3)}")
print(f"Centre      : {result_rot['centre'].round(3)}")
print(f"RMS residual: {residuals_rms(pts_rot[:,0], pts_rot[:,1], pts_rot[:,2], result_rot):.4f}")

## 5. Fitting from CSV datasets

In [ ]:
data_dir = os.path.join(os.getcwd(), "..", "data")

for fname in sorted(os.listdir(data_dir)):
    if fname.endswith(".csv"):
        path = os.path.join(data_dir, fname)
        data = np.loadtxt(path, delimiter=",", skiprows=1)
        r = fit_ellipsoid(data[:,0], data[:,1], data[:,2])
        rms = residuals_rms(data[:,0], data[:,1], data[:,2], r)
        print(f"{fname}")
        print(f"  centre : {r['centre'].round(3)}")
        print(f"  radii  : {r['radii'].round(3)}")
        print(f"  RMS    : {rms:.5f}")
        print()

## 6. Accuracy vs noise study

How does centre-estimation error grow with increasing noise?

In [ ]:
noise_levels = np.linspace(0.0, 0.5, 20)
centre_errors = []
radii_errors  = []

TRUE_C = np.array([0.0, 0.0, 0.0])
TRUE_R = np.array([4.0, 3.0, 2.0])

for sigma in noise_levels:
    p = generate_ellipsoid_points(
        centre=TRUE_C, radii=TRUE_R,
        n_points=400, noise_std=sigma, seed=0,
    )
    try:
        res = fit_ellipsoid(p[:,0], p[:,1], p[:,2])
        centre_errors.append(np.linalg.norm(res['centre'] - TRUE_C))
        radii_errors.append(np.linalg.norm(np.sort(res['radii'])[::-1] - np.sort(TRUE_R)[::-1]))
    except ValueError:
        centre_errors.append(np.nan)
        radii_errors.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(noise_levels, centre_errors, 'o-')
axes[0].set_xlabel("Noise σ"); axes[0].set_ylabel("Centre error (L2)")
axes[0].set_title("Centre recovery accuracy")
axes[1].plot(noise_levels, radii_errors, 's-', color='tab:orange')
axes[1].set_xlabel("Noise σ"); axes[1].set_ylabel("Radii error (L2)")
axes[1].set_title("Semi-axis recovery accuracy")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("noise_study.png", dpi=120)
plt.show()
print("Figure saved to noise_study.png")

## 7. Interactive 3D Visualisation

The cell below uses **Plotly** to render an interactive 3-D plot directly in the
notebook. You can rotate, zoom and pan the plot with your mouse. Toggle the
point cloud or ellipsoid surface by clicking the legend items.

In [ ]:
def make_ellipsoid_mesh(centre, radii, axes_mat, n_u=60, n_v=30):
    """Return (X, Y, Z) grid arrays for a parameterised ellipsoid surface."""
    u = np.linspace(0, 2 * np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)
    xs = radii[0] * np.outer(np.cos(u), np.sin(v))
    ys = radii[1] * np.outer(np.sin(u), np.sin(v))
    zs = radii[2] * np.outer(np.ones_like(u), np.cos(v))
    shape = xs.shape
    pts_m = np.column_stack([xs.ravel(), ys.ravel(), zs.ravel()]) @ axes_mat.T
    cx, cy, cz = centre
    X = pts_m[:, 0].reshape(shape) + cx
    Y = pts_m[:, 1].reshape(shape) + cy
    Z = pts_m[:, 2].reshape(shape) + cz
    return X, Y, Z


# Use the axis-aligned fit from Section 3
X_ell, Y_ell, Z_ell = make_ellipsoid_mesh(
    result['centre'], result['radii'], result['axes']
)

if _PLOTLY_AVAILABLE:
    fig_3d = go.Figure()

    # Point cloud
    fig_3d.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=2.5, color='steelblue', opacity=0.6),
        name='Noisy point cloud',
    ))

    # Fitted ellipsoid surface
    fig_3d.add_trace(go.Surface(
        x=X_ell, y=Y_ell, z=Z_ell,
        colorscale='Oranges',
        opacity=0.35,
        showscale=False,
        name='Fitted ellipsoid',
    ))

    fig_3d.update_layout(
        title='Li–Griffiths Ellipsoid Fit — Interactive 3D View',
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='z',
            aspectmode='data',
        ),
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, b=0, t=40),
        height=600,
    )
    fig_3d.show()
else:
    # Fallback: static matplotlib plot
    fig = plt.figure(figsize=(9, 7))
    ax  = fig.add_subplot(111, projection='3d')
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=4, alpha=0.4, label='Noisy data')
    ax.plot_surface(X_ell, Y_ell, Z_ell, alpha=0.18, color='orange')
    ax.set_title('Li–Griffiths Ellipsoid Fit')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.legend()
    plt.tight_layout()
    plt.savefig('ellipsoid_fit_vis.png', dpi=120)
    plt.show()
    print('Figure saved to ellipsoid_fit_vis.png')

## 8. Fitting Accuracy Metrics

The cells below compute detailed statistics on the algebraic residuals
`F(x, y, z) = Ax² + By² + ... + J` evaluated at every data point.
A perfect fit would give F = 0 everywhere.

In [ ]:
# --- Algebraic (residual) error statistics ---
alg_errors = algebraic_distance(x, y, z, result['coefficients'])

mean_err  = float(np.mean(alg_errors))
std_err   = float(np.std(alg_errors))
rmse      = float(np.sqrt(np.mean(alg_errors ** 2)))
max_abs   = float(np.max(np.abs(alg_errors)))

# Normalised residuals: divide by the gradient magnitude at each point
# (approximation of geometric distance denominator)
A, B, C_coef, D, E, F_coef, G, H, I_coef, J = result['coefficients']
grad_x = 2*A*x + 2*F_coef*y + 2*E*z + 2*G
grad_y = 2*F_coef*x + 2*B*y + 2*D*z + 2*H
grad_z = 2*E*x + 2*D*y + 2*C_coef*z + 2*I_coef
grad_norm = np.sqrt(grad_x**2 + grad_y**2 + grad_z**2)
norm_residuals = alg_errors / np.where(grad_norm > 1e-12, grad_norm, np.nan)

print("=" * 60)
print("  FITTING ACCURACY METRICS")
print("=" * 60)
print(f"  N points             : {len(x)}")
print()
print("  --- Algebraic residuals F(x,y,z) ---")
print(f"  Mean                 : {mean_err:+.6f}")
print(f"  Std                  :  {std_err:.6f}")
print(f"  RMSE                 :  {rmse:.6f}")
print(f"  Max |error|          :  {max_abs:.6f}")
print()
print("  --- Normalised residuals (geometric approx.) ---")
valid = ~np.isnan(norm_residuals)
print(f"  Mean                 : {np.mean(norm_residuals[valid]):+.6f}")
print(f"  Std                  :  {np.std(norm_residuals[valid]):.6f}")
print(f"  RMSE                 :  {np.sqrt(np.mean(norm_residuals[valid]**2)):.6f}")
print()
print("  --- Geometric parameters ---")
print(f"  Fitted centre        : {result['centre'].round(4)}")
print(f"  Fitted radii         : {result['radii'].round(4)}")
print(f"  True centre          : {np.array(TRUE_CENTRE)}")
print(f"  True radii           : {np.sort(TRUE_RADII)[::-1]}")
print(f"  Centre error (L2)    : {np.linalg.norm(result['centre'] - np.array(TRUE_CENTRE)):.6f}")
print(f"  Radii error (L2)     : {np.linalg.norm(np.sort(result['radii'])[::-1] - np.sort(TRUE_RADII)[::-1]):.6f}")
print("=" * 60)

In [ ]:
# Histogram of algebraic residuals
if _PLOTLY_AVAILABLE:
    fig_hist = go.Figure()
    fig_hist.add_trace(go.Histogram(
        x=alg_errors,
        nbinsx=40,
        marker_color='steelblue',
        opacity=0.75,
        name='Algebraic residuals',
    ))
    fig_hist.add_vline(x=mean_err, line_dash='dash', line_color='red',
                       annotation_text=f'Mean={mean_err:.4f}',
                       annotation_position='top right')
    fig_hist.update_layout(
        title='Distribution of Algebraic Residuals F(x,y,z)',
        xaxis_title='Algebraic residual',
        yaxis_title='Count',
        bargap=0.05,
        height=380,
    )
    fig_hist.show()
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(alg_errors, bins=40, color='steelblue', alpha=0.75)
    ax.axvline(mean_err, color='red', linestyle='--',
               label=f'Mean = {mean_err:.4f}')
    ax.set_xlabel('Algebraic residual')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Algebraic Residuals F(x,y,z)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Interactive Controls (ipywidgets)

Use the sliders and checkboxes below to explore how the fit changes
with different data parameters. The plot and metrics update automatically.

In [ ]:
def _build_plot(n_points, noise_std, show_points, show_surface, point_opacity, surface_opacity):
    """Regenerate data, re-fit, and display plot + metrics."""
    import io
    from IPython.display import clear_output
    pts_w = generate_ellipsoid_points(
        centre=TRUE_CENTRE, radii=TRUE_RADII,
        n_points=int(n_points), noise_std=float(noise_std), seed=42,
    )
    xw, yw, zw = pts_w[:, 0], pts_w[:, 1], pts_w[:, 2]
    try:
        res_w = fit_ellipsoid(xw, yw, zw)
    except ValueError as e:
        print(f"Fit failed: {e}")
        return

    # Metrics
    errs = algebraic_distance(xw, yw, zw, res_w['coefficients'])
    rms_w = float(np.sqrt(np.mean(errs ** 2)))
    c_err = np.linalg.norm(res_w['centre'] - np.array(TRUE_CENTRE))
    r_err = np.linalg.norm(np.sort(res_w['radii'])[::-1] - np.sort(TRUE_RADII)[::-1])

    # Build ellipsoid surface
    X_w, Y_w, Z_w = make_ellipsoid_mesh(
        res_w['centre'], res_w['radii'], res_w['axes']
    )

    if _PLOTLY_AVAILABLE:
        traces = []
        if show_points:
            traces.append(go.Scatter3d(
                x=pts_w[:, 0], y=pts_w[:, 1], z=pts_w[:, 2],
                mode='markers',
                marker=dict(size=2.5, color='steelblue', opacity=float(point_opacity)),
                name='Point cloud',
            ))
        if show_surface:
            traces.append(go.Surface(
                x=X_w, y=Y_w, z=Z_w,
                colorscale='Oranges',
                opacity=float(surface_opacity),
                showscale=False,
                name='Fitted ellipsoid',
            ))
        layout = go.Layout(
            title=(
                f'n={int(n_points)}, σ={float(noise_std):.3f} — '
                f'RMSE={rms_w:.5f}, '
                f'CentreΔ={c_err:.4f}, '
                f'RadiiΔ={r_err:.4f}'
            ),
            scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z',
                       aspectmode='data'),
            height=550,
            margin=dict(l=0, r=0, b=0, t=50),
        )
        go.Figure(data=traces, layout=layout).show()
    else:
        fig = plt.figure(figsize=(9, 7))
        ax  = fig.add_subplot(111, projection='3d')
        if show_points:
            ax.scatter(pts_w[:, 0], pts_w[:, 1], pts_w[:, 2],
                       s=4, alpha=float(point_opacity))
        if show_surface:
            ax.plot_surface(X_w, Y_w, Z_w,
                            alpha=float(surface_opacity), color='orange')
        ax.set_title(f'n={int(n_points)}, σ={float(noise_std):.3f} | RMSE={rms_w:.5f}')
        plt.tight_layout()
        plt.show()

    print(f"  N points  : {int(n_points)}  |  Noise σ  : {float(noise_std):.3f}")
    print(f"  RMSE      : {rms_w:.6f}")
    print(f"  Centre Δ  : {c_err:.6f}")
    print(f"  Radii Δ   : {r_err:.6f}")


if _WIDGETS_AVAILABLE:
    _slider_n = widgets.IntSlider(
        value=300, min=50, max=1000, step=50,
        description='N points:', continuous_update=False,
        layout=widgets.Layout(width='450px'),
        style={'description_width': '110px'},
    )
    _slider_noise = widgets.FloatSlider(
        value=0.05, min=0.0, max=0.5, step=0.01,
        description='Noise σ:', continuous_update=False,
        readout_format='.3f',
        layout=widgets.Layout(width='450px'),
        style={'description_width': '110px'},
    )
    _chk_points  = widgets.Checkbox(value=True,  description='Show point cloud')
    _chk_surface = widgets.Checkbox(value=True,  description='Show ellipsoid surface')
    _slider_pt_alpha  = widgets.FloatSlider(
        value=0.6, min=0.1, max=1.0, step=0.05,
        description='Point opacity:', continuous_update=False,
        readout_format='.2f',
        layout=widgets.Layout(width='450px'),
        style={'description_width': '130px'},
    )
    _slider_surf_alpha = widgets.FloatSlider(
        value=0.35, min=0.05, max=1.0, step=0.05,
        description='Surface opacity:', continuous_update=False,
        readout_format='.2f',
        layout=widgets.Layout(width='450px'),
        style={'description_width': '130px'},
    )

    _ui = widgets.VBox([
        widgets.HBox([_slider_n, _slider_noise]),
        widgets.HBox([_chk_points, _chk_surface]),
        widgets.HBox([_slider_pt_alpha, _slider_surf_alpha]),
    ])

    _out = widgets.interactive_output(
        _build_plot,
        dict(
            n_points=_slider_n,
            noise_std=_slider_noise,
            show_points=_chk_points,
            show_surface=_chk_surface,
            point_opacity=_slider_pt_alpha,
            surface_opacity=_slider_surf_alpha,
        ),
    )
    display(_ui, _out)
else:
    print("ipywidgets not available — calling _build_plot directly with default parameters.")
    _build_plot(
        n_points=300, noise_std=0.05,
        show_points=True, show_surface=True,
        point_opacity=0.6, surface_opacity=0.35,
    )

## 10. How to Interpret the Plot and Metrics

### The 3-D Plot

| Element | Description |
|---------|-------------|
| **Blue points** | Raw 3-D point cloud (the input data). Noise causes points to scatter slightly off the true surface. |
| **Orange surface** | The fitted ellipsoid surface from Li & Griffiths (2004). Ideally this passes through/near all blue points. |

- **Rotate** by clicking and dragging.
- **Zoom** with the scroll wheel.
- **Pan** by right-clicking and dragging (or Shift + drag).
- **Toggle layers** by clicking legend entries.

### The Accuracy Metrics

| Metric | What it means |
|--------|---------------|
| **Algebraic RMSE** | Root-mean-square of `F(x,y,z)` for each input point. Values ≪ 1 (relative to data scale) indicate a good fit. |
| **Mean algebraic error** | Should be near 0 for unbiased noise. A large absolute mean suggests a systematic offset. |
| **Std of errors** | Spread of algebraic residuals; grows with noise level. |
| **Normalised residuals** | Algebraic error divided by the gradient magnitude at each point — an approximation of the geometric (point-to-surface) distance. |
| **Centre Δ (L2)** | Euclidean distance between the fitted centre and the true centre (available when true parameters are known). |
| **Radii Δ (L2)** | Norm of the difference between fitted and true semi-axis lengths. |

### Rules of thumb

- A smaller algebraic RMSE always indicates a better algebraic fit, but very noisy
  data can still yield a small RMSE if the ellipsoid inflates to wrap around the noise.
- Centre and radii errors are the most interpretable accuracy measures when ground-truth
  parameters are known.
- Use the **Interactive Controls** (Section 9) to explore how fit quality degrades
  with increasing noise or insufficient data density.